# Baseline classifier — load and train

Loads `starter/baseline/baseline_classifier.py` and trains it on `starter/data/train.csv`.

In [1]:
import importlib.util
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
MODULE_PATH = Path(f"{PROJECT_ROOT}/starter/baseline/baseline_classifier.py")
DATA_PATH = Path(f"{PROJECT_ROOT}/starter/data/train.csv")

assert MODULE_PATH.exists(), MODULE_PATH
assert DATA_PATH.exists(), DATA_PATH

In [2]:
# Load the script as a module without needing it on sys.path.
spec = importlib.util.spec_from_file_location("baseline_classifier", MODULE_PATH)
baseline = importlib.util.module_from_spec(spec)
spec.loader.exec_module(baseline)

print(baseline.ROUTES)

['account-access', 'transaction-dispute', 'fraud-report', 'general']


## Data

In [3]:
from collections import Counter

texts, labels = baseline.load(DATA_PATH)
print(f"{len(texts)} rows")
Counter(labels)

400 rows


Counter({'general': 160,
         'account-access': 100,
         'transaction-dispute': 90,
         'fraud-report': 50})

## Train

Runs the module's own `main()`, which vectorises, splits 80/20 and reports accuracy.

In [4]:
acc = baseline.main()
acc

loaded 400 rows
test accuracy: 0.9875


0.9875

## Same pipeline, reproduced for inspection

`main()` returns only accuracy, so rebuild it here to reach the fitted objects.

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split

vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True)
X = vectorizer.fit_transform(texts)

X_train, X_test, y_train, y_test = train_test_split(
    X, labels, test_size=0.2, random_state=0
)

clf = LogisticRegression(max_iter=2000, C=10.0)
clf.fit(X_train, y_train)

print(f"{X.shape[1]} features, {X_train.shape[0]} train / {X_test.shape[0]} test")

1522 features, 320 train / 80 test


In [6]:
preds = clf.predict(X_test)

train_acc = accuracy_score(y_train, clf.predict(X_train))
test_acc = accuracy_score(y_test, preds)

print(f"train accuracy: {train_acc:.4f}")
print(f"test accuracy:  {test_acc:.4f}")
print(classification_report(y_test, preds, zero_division=0))

train accuracy: 1.0000
test accuracy:  0.9875
                     precision    recall  f1-score   support

     account-access       1.00      1.00      1.00        16
       fraud-report       1.00      0.93      0.97        15
            general       0.97      1.00      0.99        35
transaction-dispute       1.00      1.00      1.00        14

           accuracy                           0.99        80
          macro avg       0.99      0.98      0.99        80
       weighted avg       0.99      0.99      0.99        80



In [7]:
labels_sorted = sorted(set(labels))
cm = confusion_matrix(y_test, preds, labels=labels_sorted)

print("rows = true, cols = predicted")
print(labels_sorted)
print(cm)

rows = true, cols = predicted
['account-access', 'fraud-report', 'general', 'transaction-dispute']
[[16  0  0  0]
 [ 0 14  1  0]
 [ 0  0 35  0]
 [ 0  0  0 14]]


---

# Review of the handover

`eval_report.md` claims **98.75% accuracy** and recommends shipping. Below I check
whether that number measures what it claims to measure.

## 1. Leakage: the vectoriser is fitted before the split

In `baseline_classifier.py` the order is `fit_transform(texts)` first, `train_test_split`
second. The TF-IDF vocabulary and IDF weights are therefore learned from the test rows too.

In [8]:
import inspect

src = inspect.getsource(baseline.main)
fit_line = next(i for i, l in enumerate(src.split(chr(10))) if "fit_transform" in l)
split_line = next(i for i, l in enumerate(src.split(chr(10))) if "train_test_split" in l)

print(f"fit_transform on line {fit_line}, train_test_split on line {split_line}")
print("leakage" if fit_line < split_line else "no leakage")

fit_transform on line 6, train_test_split on line 8
leakage


In [9]:
# Correct order: fit the vectoriser on train only.
from sklearn.pipeline import make_pipeline

txt_train, txt_test, y_tr, y_te = train_test_split(
    texts, labels, test_size=0.2, random_state=0, stratify=labels
)

pipe = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True),
    LogisticRegression(max_iter=2000, C=10.0),
)
pipe.fit(txt_train, y_tr)

print(f"leaky (reported):  {acc:.4f}")
print(f"no leakage:        {accuracy_score(y_te, pipe.predict(txt_test)):.4f}")

leaky (reported):  0.9875
no leakage:        1.0000


## 2. Are the test messages actually unseen?

400 rows is small, so check how similar each test message is to its nearest training
message. Cosine similarity near 1.0 means the model has effectively seen it already.

In [10]:
print(f"rows: {len(texts)}")
print(f"unique texts: {len(set(texts))}")
print(f"exact duplicates: {len(texts) - len(set(texts))}")

rows: 400
unique texts: 400
exact duplicates: 0


In [11]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

v = TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True).fit(txt_train)
sim = cosine_similarity(v.transform(txt_test), v.transform(txt_train))
nearest = sim.max(axis=1)

for t in (0.99, 0.95, 0.90, 0.80):
    print(f"test rows with a train neighbour >= {t:.2f}: {(nearest >= t).sum():3d} / {len(nearest)}")
print(f"median nearest-neighbour similarity: {np.median(nearest):.3f}")

test rows with a train neighbour >= 0.99:   1 / 80
test rows with a train neighbour >= 0.95:  11 / 80
test rows with a train neighbour >= 0.90:  26 / 80
test rows with a train neighbour >= 0.80:  62 / 80
median nearest-neighbour similarity: 0.878


In [12]:
# The data is templated: same sentence, different asset name.
import re

def template(t):
    t = re.sub(r"[0-9]+(\.[0-9]+)?", " NUM ", t.lower())
    t = re.sub(r"[^a-z ]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

tmpl = [template(t) for t in texts]
print(f"unique templates: {len(set(tmpl))} for {len(texts)} rows")
for s, n in Counter(tmpl).most_common(5):
    print(f"  {n:3d}  {s[:70]}")

unique templates: 398 for 400 rows
    2  hello team i bought of usdt but the order shows a different price than
    2  my withdrawal of shows completed but i never received it please revers
    1  how does staking work and what rewards can i expect on polygon please 
    1  hello team the app won t let me sign in it just spins on the login scr
    1  hey my withdrawal of eth shows completed but i never received it pleas


Removing greetings, numbers and asset names collapses the corpus much further: the
messages are generated from a small set of intent templates with slots filled in.

In [13]:
ASSETS = set(
    "polygon ethereum bitcoin solana arbitrum avalanche cardano dogecoin eth btc sol "
    "usdc usdt matic ada doge bnb xrp litecoin ltc optimism base tron stellar xlm dot "
    "polkadot".split()
)
GREETINGS = ["hello team", "hi team", "hey team", "hello", "hi there", "hi", "hey",
             "good morning", "good afternoon", "dear support", "dear team"]
SIGNOFFS = ["thanks in advance", "appreciate any help", "any help appreciated",
            "please advise", "thank you", "thanks", "best regards", "regards",
            "many thanks", "cheers"]


def intent(text):
    """Reduce a message to its intent template: drop slots, greetings and sign-offs."""
    t = re.sub(r"[0-9]+(\.[0-9]+)?", " N ", text.lower())
    t = re.sub(r"[^a-z ]", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    for g in GREETINGS:
        if t.startswith(g + " "):
            t = t[len(g) + 1:]
            break
    for s in SIGNOFFS:
        if t.endswith(" " + s):
            t = t[: -(len(s) + 1)]
            break
    return " ".join(w for w in t.split() if w not in ASSETS)


groups = [intent(t) for t in texts]
print(f"{len(set(groups))} intent templates for {len(texts)} messages")
for s, n in Counter(groups).most_common(6):
    print(f"  {n:3d}  {s[:65]}")

256 intent templates for 400 messages
    9  how do i enable price alerts for
    8  how long do withdrawals usually take to process
    7  can you explain how to move to an external wallet
    6  how does staking work and what rewards can i expect on
    6  where can i download my tax documents for last year
    6  a transaction has been stuck in pending for two days and the fund


## 3. The honest number: group the split by template

If a template appears in training, its siblings in the test set are not a real test.
`StratifiedGroupKFold` keeps every template entirely on one side of the split, so the
model is scored on intents it has never seen.

In [14]:
from sklearn.model_selection import (
    StratifiedGroupKFold,
    cross_val_predict,
    cross_val_score,
)

def fresh_pipe():
    return make_pipeline(
        TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True),
        LogisticRegression(max_iter=2000, C=10.0),
    )

X_txt = np.array(texts)
y = np.array(labels)
g = np.array(groups)

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0)
grouped_pred = cross_val_predict(fresh_pipe(), X_txt, y, groups=g, cv=sgkf)
grouped_acc = accuracy_score(y, grouped_pred)

random_cv = cross_val_score(fresh_pipe(), X_txt, y, cv=5, scoring="accuracy").mean()

print(f"reported (leaky split):        {acc:.4f}")
print(f"random 5-fold CV:              {random_cv:.4f}")
print(f"grouped 5-fold CV (honest):    {grouped_acc:.4f}")

reported (leaky split):        0.9875
random 5-fold CV:              1.0000
grouped 5-fold CV (honest):    1.0000


In [15]:
print(classification_report(y, grouped_pred, zero_division=0))

                     precision    recall  f1-score   support

     account-access       1.00      1.00      1.00       100
       fraud-report       1.00      1.00      1.00        50
            general       1.00      1.00      1.00       160
transaction-dispute       1.00      1.00      1.00        90

           accuracy                           1.00       400
          macro avg       1.00      1.00      1.00       400
       weighted avg       1.00      1.00      1.00       400



Grouped CV also returns 1.0000. The conclusion is not that the model is excellent — it is
that **this dataset cannot distinguish a good router from a mediocre one**. The next cell
shows how little signal is actually needed to score well.

In [16]:
# If a 20-word vocabulary nearly matches the full model, the task is keyword lookup.
for n in (5, 10, 20, 30, None):
    m = make_pipeline(
        TfidfVectorizer(max_features=n, ngram_range=(1, 2), sublinear_tf=True),
        LogisticRegression(max_iter=2000, C=10.0),
    )
    s = cross_val_score(m, X_txt, y, cv=5, scoring="accuracy").mean()
    print(f"vocab {str(n or 'full'):>4}: {s:.4f}")

vocab    5: 0.5250
vocab   10: 0.7400
vocab   20: 0.8325


vocab   30: 0.8475


vocab full: 1.0000


## 4. Behaviour on messages that are not from the template set

Production traffic will not be templated. These probes are paraphrases and mixed-intent
messages written by hand, so they test what the held-out score cannot.

In [17]:
full = fresh_pipe().fit(X_txt, y)

probes = [
    ("someone got into my account and moved my funds out", "fraud-report"),
    ("i think my account was hacked, unauthorized withdrawal", "fraud-report"),
    ("there is a withdrawal i did not authorise, please freeze everything", "fraud-report"),
    ("i got a text pretending to be you asking for my seed phrase", "fraud-report"),
    ("scammer tricked me into sending funds", "fraud-report"),
    ("I can't log in AND I see a transfer I never made", "fraud-report"),
    ("cant login", "account-access"),
    ("charged twice", "transaction-dispute"),
    ("what are your fees", "general"),
]

texts_p = [t for t, _ in probes]
pred_p = full.predict(texts_p)
conf_p = full.predict_proba(texts_p).max(axis=1)

for (t, expected), pr, cf in zip(probes, pred_p, conf_p):
    flag = "ok  " if pr == expected else "MISS"
    print(f"{flag} {pr:<20} conf={cf:.2f}  expected={expected:<20} {t[:48]}")

print(f"{(pred_p == [e for _, e in probes]).sum()}/{len(probes)} correct")

ok   fraud-report         conf=0.85  expected=fraud-report         someone got into my account and moved my funds o
ok   fraud-report         conf=0.88  expected=fraud-report         i think my account was hacked, unauthorized with
ok   fraud-report         conf=0.52  expected=fraud-report         there is a withdrawal i did not authorise, pleas
ok   fraud-report         conf=0.40  expected=fraud-report         i got a text pretending to be you asking for my 
ok   fraud-report         conf=0.49  expected=fraud-report         scammer tricked me into sending funds
MISS account-access       conf=0.46  expected=fraud-report         I can't log in AND I see a transfer I never made
ok   account-access       conf=0.83  expected=account-access       cant login
ok   transaction-dispute  conf=0.73  expected=transaction-dispute  charged twice
ok   general              conf=0.93  expected=general              what are your fees
8/9 correct


## 5. The metric that matters: missed fraud

Routing a fraud report to `general` delays a time-critical case. The reverse error only
costs an analyst's time. These are not equal, so accuracy is the wrong headline metric —
**fraud-report recall** is.

In [18]:
FRAUD = "fraud-report"

rec = recall_score(y, grouped_pred, labels=[FRAUD], average="macro")
prec = precision_score(y, grouped_pred, labels=[FRAUD], average="macro")
print(f"fraud-report recall on templated data:    {rec:.4f}")
print(f"fraud-report precision on templated data: {prec:.4f}")

# On the hand-written probes, which is the case production resembles:
fraud_probes = [(t, e) for t, e in probes if e == FRAUD]
fp_pred = full.predict([t for t, _ in fraud_probes])
caught = (fp_pred == FRAUD).sum()
print(f"fraud caught on paraphrased probes:       {caught}/{len(fraud_probes)}")

fraud-report recall on templated data:    1.0000
fraud-report precision on templated data: 1.0000
fraud caught on paraphrased probes:       5/6


In [19]:
# Confidence on fraud messages is low, so a review threshold is cheap to add.
fraud_conf = [c for (t, e), c in zip(probes, conf_p) if e == FRAUD]
other_conf = [c for (t, e), c in zip(probes, conf_p) if e != FRAUD]

print(f"mean confidence, fraud probes: {np.mean(fraud_conf):.2f}")
print(f"mean confidence, other probes: {np.mean(other_conf):.2f}")

for th in (0.5, 0.6, 0.7):
    n = (conf_p < th).sum()
    print(f"threshold {th}: {n}/{len(probes)} messages would go to human review")

mean confidence, fraud probes: 0.60
mean confidence, other probes: 0.83
threshold 0.5: 3/9 messages would go to human review
threshold 0.6: 4/9 messages would go to human review
threshold 0.7: 4/9 messages would go to human review


In [20]:
# Recall-first alternative: bias the decision toward fraud-report.
weighted = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True),
    LogisticRegression(max_iter=2000, C=10.0, class_weight={FRAUD: 5.0}),
)
wp = cross_val_predict(weighted, X_txt, y, groups=g, cv=sgkf)

print(f"baseline  fraud recall: {recall_score(y, grouped_pred, labels=[FRAUD], average='macro'):.4f}")
print(f"weighted  fraud recall: {recall_score(y, wp, labels=[FRAUD], average='macro'):.4f}")
print(f"baseline  macro F1: {f1_score(y, grouped_pred, average='macro'):.4f}")
print(f"weighted  macro F1: {f1_score(y, wp, average='macro'):.4f}")

baseline  fraud recall: 1.0000
weighted  fraud recall: 1.0000
baseline  macro F1: 1.0000
weighted  macro F1: 1.0000
